In [2]:
import logging
import time
import pandas as pd
from habanero import Crossref
cr = Crossref()
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type

retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([([ConnectionError, TimeoutError])])
)()

logging.basicConfig(
    filename='DOI.log',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

start_time = time.time()

csv_file = 'pmid_journal_doi_sample.csv'       #get csv file
df = pd.read_csv(csv_file)
dois = df.iloc[:, 2].tolist() # get dois from column (3rd in this case)

def publisher_crossref_doi(dois):
    publishers = []
    for doi in dois:
        try:
            work = cr.works(ids=doi)
            publisher = work["message"].get("publisher")
            publishers.append(publisher)
        except Exception as e:
            logging.error(f"Error: API request failed for {doi}: {e}")
            publishers.append(None)
    return publishers

# Call function and assign result to a list
publisher_list = publisher_crossref_doi(dois)

# Add the publisher_list to the DataFrame as a new column
df['publisher'] = publisher_list

# Save the updated DataFrame to a new CSV file
df.to_csv('pmid_doi_journal_publisher.csv', index=False)
logging.info("Done")